# Train the ONE model (YOLO11) — Day 3

Run this in **Google Colab** with a **free GPU**.

1. Upload this notebook to https://colab.research.google.com
2. **Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU -> Save**
3. Run the cells top to bottom.

Output: `best.pt` (your trained model) + metric plots. Download `best.pt` into the
repo at `models/best.pt`, then run the Streamlit app locally.

## 1. Install + confirm GPU

In [ ]:
!pip -q install ultralytics roboflow
import torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
!nvidia-smi -L  # should list a Tesla T4. If error: set Runtime -> T4 GPU.

## 2. Download your unified dataset from Roboflow

Paste the **export snippet** Roboflow gave you (BUILD_GUIDE.md, Chapter 3). It looks like the
cell below — replace API key / workspace / project / version with yours.

In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_KEY")
project = rf.workspace("YOUR_WS").project("drawing-checklist")
dataset = project.version(1).download("yolov11")

DATA_YAML = f"{dataset.location}/data.yaml"
print('data.yaml ->', DATA_YAML)
!cat {DATA_YAML}  # CONFIRM names order matches config.yaml

## 3. Train

Knobs explained in BUILD_GUIDE.md Chapter 13. Start with these; raise `epochs` if loss still
falling, lower `batch` if you hit out-of-memory.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # nano: fast, fine for a demo. Try 'yolo11s.pt' if time.
results = model.train(
    data=DATA_YAML,
    epochs=80,
    imgsz=960,      # match Roboflow resize
    batch=16,       # lower to 8 if OOM
    patience=20,    # early-stop if no improvement for 20 epochs
    project='runs', name='drawing', exist_ok=True,
)

## 4. Validate + read the metrics

Look at `mAP50` (overall) and per-class numbers. Weak classes (e.g. crop-style
`gdt_symbol`/`surface_roughness`) -> consider dropping them from `config.yaml` or
hand-labelling more full sheets (Day 9). Screenshot the plots for your slides.

In [ ]:
metrics = model.val()
print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)

from IPython.display import Image, display
for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
    p = f'runs/drawing/{f}'
    import os
    if os.path.exists(p):
        print(p); display(Image(filename=p))

## 5. Sanity-check on a test image

In [ ]:
import glob, os
from IPython.display import Image, display

best = 'runs/drawing/weights/best.pt'
m = YOLO(best)
test_imgs = glob.glob(f'{dataset.location}/test/images/*')[:3]
for img in test_imgs:
    r = m.predict(img, conf=0.25, save=True, verbose=False)
    print(os.path.basename(img), '->', [m.names[int(b.cls[0])] for b in r[0].boxes])
    display(Image(filename=r[0].save_dir + '/' + os.path.basename(img)))

## 6. Download best.pt

Run the cell, then move the downloaded file to `models/best.pt` in the repo. Re-run the
Streamlit app locally and the checklist comes alive.

In [ ]:
from google.colab import files
files.download('runs/drawing/weights/best.pt')